# HQDE Single-Node A100 Scheduled Benchmark

Self-contained notebook. No extra helper file required.

In [ ]:
%pip install -q torch torchvision ray pandas matplotlib hqde


In [ ]:
import copy, json, random, time
from contextlib import nullcontext
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np, pandas as pd, ray, torch, torch.nn as nn, torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

OUT = Path.cwd() / 'benchmark_outputs' / 'single_node_a100'
OUT.mkdir(parents=True, exist_ok=True)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

MEANS = {
    'cifar10': ((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
    'cifar100': ((0.5071,0.4867,0.4408),(0.2675,0.2565,0.2761)),
    'svhn': ((0.4377,0.4438,0.4728),(0.1980,0.2010,0.1970)),
}

@dataclass
class DatasetSpec:
    name: str
    data_root: str = './data'
    subset_size: int | None = 20000
    train_batch_size: int = 128
    eval_batch_size: int = 256
    actor_dataloader_workers: int = 2

@dataclass
class ScheduledHQDEConfig:
    logical_workers: int = 12
    active_gpu_executors: int = 4
    gpu_fraction_per_executor: float = 0.25
    local_epochs: int = 1
    global_rounds: int = 20
    learning_rate: float = 1e-3
    weight_decay: float = 5e-4
    label_smoothing: float = 0.1
    use_amp: bool = True
    compile_model: bool = False
    compile_mode: str = 'default'
    aggregation_mode: str = 'efficiency_weighted'
    use_quantization: bool = True
    quantization_config: dict | None = None
    partition_mode: str = 'dirichlet'
    dirichlet_alpha: float = 0.5
    seed: int = 42
    num_cpus: int = 12

class SmallImageResNet18(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.0):
        super().__init__()
        m = torchvision.models.resnet18(num_classes=num_classes)
        m.conv1 = nn.Conv2d(3,64,3,1,1,bias=False)
        m.maxpool = nn.Identity()
        if dropout_rate > 0:
            m.fc = nn.Sequential(nn.Dropout(dropout_rate), nn.Linear(m.fc.in_features, num_classes))
        self.model = m
    def forward(self, x): return self.model(x)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def tfms(name, train):
    mean, std = MEANS[name]
    if train:
        return transforms.Compose([transforms.RandomCrop(32,padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(mean,std)])
    return transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean,std)])

def load_dataset(name, root, train, subset_size, seed):
    name = name.lower()
    t = tfms(name, train)
    if name == 'cifar10':
        ds = torchvision.datasets.CIFAR10(root=root, train=train, download=True, transform=t); nc = 10
    elif name == 'cifar100':
        ds = torchvision.datasets.CIFAR100(root=root, train=train, download=True, transform=t); nc = 100
    else:
        split = 'train' if train else 'test'; ds = torchvision.datasets.SVHN(root=root, split=split, download=True, transform=t); nc = 10
    if subset_size is not None and subset_size < len(ds):
        idx = np.random.default_rng(seed + (0 if train else 1)).choice(len(ds), size=int(subset_size), replace=False).tolist()
        ds = Subset(ds, idx)
    return ds, nc

def labels_of(ds):
    if isinstance(ds, Subset): return labels_of(ds.dataset)[np.asarray(ds.indices)]
    vals = ds.targets if hasattr(ds, 'targets') else ds.labels
    return np.asarray(vals, dtype=np.int64)

def create_loaders(spec, seed=42):
    tr, nc = load_dataset(spec.name, spec.data_root, True, spec.subset_size, seed)
    te, _ = load_dataset(spec.name, spec.data_root, False, spec.subset_size, seed)
    tl = DataLoader(tr, batch_size=spec.train_batch_size, shuffle=True, num_workers=2, pin_memory=True)
    vl = DataLoader(te, batch_size=spec.eval_batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return tr, te, tl, vl, nc

def partition_indices(labels, n, mode='dirichlet', alpha=0.5, seed=42):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels, dtype=np.int64)
    if mode == 'iid': return [x.tolist() for x in np.array_split(rng.permutation(len(labels)), n) if len(x) > 0]
    worker_indices = [[] for _ in range(n)]
    for c in range(int(labels.max()) + 1):
        idx = np.where(labels == c)[0].copy(); rng.shuffle(idx)
        props = rng.dirichlet(np.full(n, alpha, dtype=np.float64))
        cuts = (np.cumsum(props) * len(idx)).astype(int)[:-1]
        for wid, split in enumerate(np.split(idx, cuts)): worker_indices[wid].extend(split.tolist())
    for x in worker_indices: rng.shuffle(x)
    return worker_indices

class AdaptiveQuantizer:
    def __init__(self, base_bits=8, min_bits=4, max_bits=16): self.base_bits, self.min_bits, self.max_bits = base_bits, min_bits, max_bits
    def compute_importance_score(self, w):
        imp = torch.abs(w)
        if imp.numel() == 0: return torch.zeros_like(imp)
        mn, mx = imp.min(), imp.max()
        return (imp - mn) / (mx - mn) if mx > mn else torch.ones_like(imp) * 0.5
    def adaptive_quantize(self, w, imp):
        bits = torch.clamp(self.min_bits + (self.max_bits - self.min_bits) * imp, self.min_bits, self.max_bits).int()
        avg_bits = int(bits.float().mean().item())
        mn, mx = w.min(), w.max()
        if mx > mn:
            scale = (mx - mn) / (2 ** avg_bits - 1)
            q = torch.round((w - mn) / scale).clamp(0, 2 ** avg_bits - 1)
            dq = q * scale + mn
        else:
            dq = w.clone()
        return dq, {'compression_ratio': 32.0 / avg_bits}

class QuantumInspiredAggregator:
    def efficiency_weighted_aggregation(self, vals, scores):
        dev = vals[0].device
        x = torch.stack([v.to(dev) for v in vals], 0)
        w = torch.softmax(torch.tensor(scores, dtype=torch.float32, device=dev), 0)
        dims = (w.shape[0],) + (1,) * (x.dim() - 1)
        return (x * w.view(dims)).sum(0)

class LogicalShardWorker:
    def __init__(self, worker_id, shard_indices): self.worker_id, self.shard_indices, self.history = int(worker_id), list(map(int, shard_indices)), []
    def get_assignment(self): return {'worker_id': self.worker_id, 'indices': self.shard_indices}
    def record_round(self, metrics): self.history.append(dict(metrics)); return True
    def get_history(self): return list(self.history)

class GPUExecutionWorker:
    def __init__(self, dataset_name, data_root, subset_size, seed, actor_dataloader_workers=2):
        self.train_dataset, self.num_classes = load_dataset(dataset_name, data_root, True, subset_size, seed)
        self.test_dataset, _ = load_dataset(dataset_name, data_root, False, subset_size, seed)
        self.actor_dataloader_workers = int(actor_dataloader_workers)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    def _loader(self, indices, batch_size, shuffle):
        return DataLoader(Subset(self.train_dataset, list(indices)), batch_size=batch_size, shuffle=shuffle, num_workers=self.actor_dataloader_workers, pin_memory=True, persistent_workers=self.actor_dataloader_workers > 0)
    def train_local(self, state, shard_indices, local_epochs, batch_size, lr, wd, ls, use_amp, compile_model, compile_mode):
        model = SmallImageResNet18(num_classes=self.num_classes).to(self.device)
        model.load_state_dict(state, strict=True)
        if self.device.type == 'cuda': model = model.to(memory_format=torch.channels_last)
        if compile_model and hasattr(torch, 'compile'): model = torch.compile(model, mode=str(compile_mode))
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        crit = nn.CrossEntropyLoss(label_smoothing=ls)
        amp = bool(use_amp and self.device.type == 'cuda')
        scaler = torch.cuda.amp.GradScaler(enabled=amp) if amp else None
        total_samples = total_correct = 0; total_loss = 0.0
        model.train(); loader = self._loader(shard_indices, batch_size, True)
        for _ in range(int(local_epochs)):
            for xb, yb in loader:
                xb, yb = xb.to(self.device, non_blocking=True), yb.to(self.device, non_blocking=True)
                if self.device.type == 'cuda' and xb.ndim == 4: xb = xb.contiguous(memory_format=torch.channels_last)
                opt.zero_grad(set_to_none=True)
                ctx = torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True) if amp else nullcontext()
                with ctx: out = model(xb); loss = crit(out, yb)
                if amp:
                    scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
                else:
                    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
                with torch.no_grad():
                    pred = out.argmax(1); bs = int(yb.size(0)); total_samples += bs; total_correct += int((pred == yb).sum().item()); total_loss += float(loss.item()) * bs
        avg_loss = total_loss / max(total_samples, 1); avg_acc = total_correct / max(total_samples, 1)
        eff = 0.9 + 0.1 * max(avg_acc, 1.0 / (1.0 + avg_loss))
        return {'worker_state': {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}, 'num_samples': total_samples, 'loss': avg_loss, 'accuracy': avg_acc, 'efficiency_score': eff}
    def evaluate(self, state, batch_size):
        model = SmallImageResNet18(num_classes=self.num_classes).to(self.device)
        model.load_state_dict(state, strict=True)
        if self.device.type == 'cuda': model = model.to(memory_format=torch.channels_last)
        loader = DataLoader(self.test_dataset, batch_size=batch_size, shuffle=False, num_workers=self.actor_dataloader_workers, pin_memory=True, persistent_workers=self.actor_dataloader_workers > 0)
        crit = nn.CrossEntropyLoss(); total_samples = total_correct = 0; total_loss = 0.0; model.eval()
        with torch.no_grad():
            for xb, yb in loader:
                xb, yb = xb.to(self.device, non_blocking=True), yb.to(self.device, non_blocking=True)
                if self.device.type == 'cuda' and xb.ndim == 4: xb = xb.contiguous(memory_format=torch.channels_last)
                out = model(xb); loss = crit(out, yb); pred = out.argmax(1); bs = int(yb.size(0))
                total_samples += bs; total_correct += int((pred == yb).sum().item()); total_loss += float(loss.item()) * bs
        return {'loss': total_loss / max(total_samples, 1), 'accuracy': total_correct / max(total_samples, 1)}

def build_initial_state(num_classes):
    m = SmallImageResNet18(num_classes=num_classes)
    return {n: t.detach().cpu().clone() for n, t in m.state_dict().items()}

def weighted_mean(vals, weights):
    denom = float(sum(weights)) or 1.0; out = None
    for v, w in zip(vals, weights): out = v.float() * (w / denom) if out is None else out + v.float() * (w / denom)
    return out

def aggregate_worker_states(results, mode='efficiency_weighted', use_quantization=False, quantization_config=None):
    agg, quant = QuantumInspiredAggregator(), AdaptiveQuantizer(**(quantization_config or {})) if use_quantization else None
    sample_weights = [float(r['num_samples']) for r in results]
    scores = [float(r['efficiency_score']) * max(float(r['num_samples']), 1.0) for r in results]
    state0, out, comp = results[0]['worker_state'], {}, []
    for name, ref in state0.items():
        vals = [r['worker_state'][name] for r in results]
        if not torch.is_floating_point(ref): out[name] = vals[0].clone(); continue
        prep = []
        for v in vals:
            x = v.float()
            if quant is not None:
                x, meta = quant.adaptive_quantize(x, quant.compute_importance_score(x)); comp.append(float(meta['compression_ratio']))
            prep.append(x)
        out[name] = (weighted_mean(prep, sample_weights) if mode == 'mean' else agg.efficiency_weighted_aggregation(prep, scores).cpu()).to(dtype=ref.dtype)
    return out, {'compression_ratio': float(np.mean(comp)) if comp else 1.0, 'avg_worker_accuracy': float(np.mean([r['accuracy'] for r in results])), 'avg_worker_loss': float(np.mean([r['loss'] for r in results]))}

def ensure_ray(cfg):
    if ray.is_initialized(): return
    ray.init(ignore_reinit_error=True, log_to_driver=False, num_cpus=int(cfg.num_cpus), num_gpus=(1 if torch.cuda.is_available() else 0))

def run_scheduled_hqde_experiment(spec, cfg):
    seed_everything(cfg.seed); ensure_ray(cfg)
    train_ds, _, _, _, num_classes = create_loaders(spec, cfg.seed)
    parts = partition_indices(labels_of(train_ds), cfg.logical_workers, cfg.partition_mode, cfg.dirichlet_alpha, cfg.seed)
    Logical = ray.remote(num_cpus=0.1)(LogicalShardWorker)
    GPU = ray.remote(num_gpus=(cfg.gpu_fraction_per_executor if torch.cuda.is_available() else 0.0))(GPUExecutionWorker)
    logical = [Logical.remote(i, idx) for i, idx in enumerate(parts)]
    gpu = [GPU.remote(spec.name, spec.data_root, spec.subset_size, cfg.seed + i, spec.actor_dataloader_workers) for i in range(cfg.active_gpu_executors)]
    assign = ray.get([w.get_assignment.remote() for w in logical]); state = build_initial_state(num_classes); hist = []; started = time.time()
    for rnd in range(cfg.global_rounds):
        t0 = time.time(); state_ref = ray.put(state); results = []
        for start in range(0, len(assign), cfg.active_gpu_executors):
            wave = assign[start:start + cfg.active_gpu_executors]
            fut = [gpu[i].train_local.remote(state_ref, a['indices'], cfg.local_epochs, spec.train_batch_size, cfg.learning_rate, cfg.weight_decay, cfg.label_smoothing, cfg.use_amp, cfg.compile_model, cfg.compile_mode) for i, a in enumerate(wave)]
            vals = ray.get(fut); results.extend(vals)
            for a, r in zip(wave, vals): logical[a['worker_id']].record_round.remote({'round': rnd + 1, 'loss': r['loss'], 'accuracy': r['accuracy'], 'num_samples': r['num_samples']})
        state, meta = aggregate_worker_states(results, cfg.aggregation_mode, cfg.use_quantization, cfg.quantization_config)
        ev = ray.get(gpu[0].evaluate.remote(ray.put(state), spec.eval_batch_size))
        hist.append({'round': rnd + 1, 'eval_accuracy': float(ev['accuracy']), 'eval_loss': float(ev['loss']), 'avg_worker_accuracy': float(meta['avg_worker_accuracy']), 'avg_worker_loss': float(meta['avg_worker_loss']), 'compression_ratio': float(meta['compression_ratio']), 'round_time_sec': time.time() - t0})
    return {'dataset': spec.name, 'num_classes': num_classes, 'config': asdict(cfg), 'dataset_config': asdict(spec), 'final_accuracy': hist[-1]['eval_accuracy'] if hist else 0.0, 'final_loss': hist[-1]['eval_loss'] if hist else 0.0, 'training_time_sec': time.time() - started, 'round_history': hist, 'logical_worker_histories': ray.get([w.get_history.remote() for w in logical])}

def run_standard_hqde_baseline(spec, num_epochs=20, num_workers=4, seed=42):
    try:
        from hqde import create_hqde_system
    except Exception as exc:
        return {'available': False, 'reason': f'hqde import failed: {exc}'}
    seed_everything(seed); _, _, train_loader, test_loader, nc = create_loaders(spec, seed)
    system = create_hqde_system(model_class=SmallImageResNet18, model_kwargs={'num_classes': nc}, num_workers=num_workers)
    try:
        t0 = time.time(); metrics = system.train(train_loader, num_epochs=num_epochs); preds = system.predict(test_loader)
        ys = torch.cat([yb for _, yb in test_loader], 0); acc = float((preds.argmax(1) == ys).float().mean().item())
        return {'available': True, 'dataset': spec.name, 'num_workers': num_workers, 'final_accuracy': acc, 'training_time_sec': time.time() - t0, 'metrics': copy.deepcopy(metrics)}
    finally:
        try: system.cleanup()
        except Exception: pass

def save_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2), encoding='utf-8')

def summarize_rounds(rows):
    return pd.DataFrame([{'round': r['round'], 'eval_accuracy': round(r['eval_accuracy'],4), 'eval_loss': round(r['eval_loss'],4), 'avg_worker_accuracy': round(r['avg_worker_accuracy'],4), 'compression_ratio': round(r['compression_ratio'],3), 'round_time_sec': round(r['round_time_sec'],2)} for r in rows])

print('Notebook is self-contained.')


In [ ]:
DATASETS = [
    DatasetSpec(name='cifar10', data_root=str(Path.cwd() / 'data'), subset_size=20000, train_batch_size=128, eval_batch_size=256, actor_dataloader_workers=2),
    DatasetSpec(name='cifar100', data_root=str(Path.cwd() / 'data'), subset_size=20000, train_batch_size=128, eval_batch_size=256, actor_dataloader_workers=2),
]

CFG = ScheduledHQDEConfig(
    logical_workers=12,
    active_gpu_executors=4,
    gpu_fraction_per_executor=0.25,
    local_epochs=1,
    global_rounds=20,
    learning_rate=1e-3,
    weight_decay=5e-4,
    label_smoothing=0.1,
    use_amp=True,
    compile_model=False,
    aggregation_mode='efficiency_weighted',
    use_quantization=True,
    quantization_config={'base_bits': 8, 'min_bits': 4, 'max_bits': 16},
    partition_mode='dirichlet',
    dirichlet_alpha=0.5,
    seed=42,
    num_cpus=12,
)
RUN_STANDARD_BASELINE = True
BASELINE_EPOCHS = 20
BASELINE_WORKERS = 4
CFG


In [ ]:
scheduled_results, summaries = {}, {}
for spec in DATASETS:
    print(f'\nRunning scheduled HQDE on {spec.name} ...')
    res = run_scheduled_hqde_experiment(spec, CFG)
    scheduled_results[spec.name] = res
    summaries[spec.name] = summarize_rounds(res['round_history'])
    save_json(res, OUT / f'{spec.name}_scheduled_hqde.json')
    print('Final accuracy:', round(res['final_accuracy'], 4))


In [ ]:
baseline_results = {}
if RUN_STANDARD_BASELINE:
    for spec in DATASETS:
        print(f'\nRunning standard HQDE baseline on {spec.name} ...')
        res = run_standard_hqde_baseline(spec, num_epochs=BASELINE_EPOCHS, num_workers=BASELINE_WORKERS, seed=CFG.seed)
        baseline_results[spec.name] = res
        save_json(res, OUT / f'{spec.name}_standard_hqde.json')
        print(res if not res.get('available', False) else {'dataset': res['dataset'], 'final_accuracy': round(res['final_accuracy'], 4), 'training_time_sec': round(res['training_time_sec'], 2)})


In [ ]:
rows = []
for name, res in scheduled_results.items():
    rows.append({'dataset': name, 'experiment': 'scheduled_hqde_12logical_4gpu', 'final_accuracy': res['final_accuracy'], 'final_loss': res['final_loss'], 'training_time_sec': res['training_time_sec']})
for name, res in baseline_results.items():
    if res.get('available', False):
        rows.append({'dataset': name, 'experiment': f"standard_hqde_{res['num_workers']}w", 'final_accuracy': res['final_accuracy'], 'final_loss': res['metrics'].get('final_loss', None), 'training_time_sec': res['training_time_sec']})
summary_df = pd.DataFrame(rows).sort_values(['dataset', 'experiment'])
summary_df


In [ ]:
import matplotlib.pyplot as plt
name = DATASETS[0].name
frame = summaries[name]
display(frame.tail(10))
plt.figure(figsize=(8,4))
plt.plot(frame['round'], frame['eval_accuracy'], marker='o', label='Eval Accuracy')
plt.plot(frame['round'], frame['avg_worker_accuracy'], marker='x', label='Avg Worker Accuracy')
plt.title(f'Scheduled HQDE on {name}')
plt.xlabel('Round')
plt.ylabel('Accuracy')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:
save_json({'scheduled_results': scheduled_results, 'baseline_results': baseline_results, 'table': rows}, OUT / 'combined_summary.json')
print('Saved to', OUT)
if ray.is_initialized(): ray.shutdown()
